In [2]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# Smart repo root finder — works regardless of where notebook lives
def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'portfolio_toolkit').exists()

def find_repo_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for path in candidates:
        if is_repo_root(path):
            return path
    raise RuntimeError("Could not find repo root. Make sure you're running from inside the repo.")

repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)

from portfolio_toolkit import (
    load_prices,
    build_features,
    validate_feature_frame,
    make_forward_alpha_target,
    slice_split,
    validate_prediction_frame,
    weights_from_predictions_risk_adjusted,
    baseline_weights,
    backtest_weights,
    write_backtest_artifacts,
    build_metrics,
    init_mlflow,
    start_run,
    log_predictions,
    log_portfolio,
    log_backtest,
    log_report_artifacts,
)

print("Imports successful.")

repo_root = C:\Users\brixn\Documents\Portfolio-Optimization-Lib
Imports successful.


In [3]:
# Going to use shared_set_1 for more training
dataset_name = "shared_set_1"
prices = load_prices(dataset_name, repo_root=repo_root)
print("Prices loaded:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())

Prices loaded: (1463605, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00


In [ ]:
def add_custom_features(features: pd.DataFrame) -> pd.DataFrame:
    """
    Custom notebook-local features built on top of toolkit features.
    All features use only past data — no future information leakage.
    """
    # Risk-adjusted momentum: how strong is momentum relative to volatility
    features["mom_vol_ratio"] = (
        features["momentum_20d"] / features["vol_20d"].replace(0, np.nan)
    )
    # Downside risk ratio: what fraction of total vol is on the downside
    features["downside_ratio"] = (
        features["downside_vol_20d"] / features["vol_20d"].replace(0, np.nan)
    )
    # Momentum divergence: short term vs long term momentum shift
    features["mom_divergence"] = (
        features["momentum_5d"] - features["momentum_20d"]
    )
    # Beta-adjusted momentum: rewards strong returns from low-beta stocks
    features["mom_beta_adjusted"] = (
        features["momentum_20d"] / features["beta_20d_spy"].replace(0, np.nan)
    )
    # Trend strength: combines price trend and relative performance
    features["trend_strength"] = (
        features["price_to_sma_20d"] * features["momentum_20d"]
    )
    return features

# Build 12 toolkit features
features = build_features(prices, feature_names=[
    "momentum_5d",
    "momentum_20d",
    "vol_20d",
    "downside_vol_20d",
    "beta_20d_spy",
    "price_to_sma_20d",
    "rsi_14",
    "macd_hist",
    "bollinger_z_20d",
    "volume_zscore_20d",
    "excess_return_5d_vs_spy",
    "excess_return_20d_vs_spy",
])

# Add 5 custom features
features = add_custom_features(features)
features = validate_feature_frame(features)

# Target: forward alpha vs SPY (not raw return)
# Forces model to predict winners vs losers, not just "stocks go up"
target_alpha = make_forward_alpha_target(prices, horizon=5)

dataset = features.merge(target_alpha, on=["date", "ticker"], how="left").dropna()
print("Dataset shape:", dataset.shape)
print("Total features:", len([c for c in dataset.columns if c not in ["date", "ticker", "forward_alpha_5d_vs_spy"]]))

KeyboardInterrupt: 

In [1]:
train = slice_split(dataset, dataset_name, "train", repo_root=repo_root)
val   = slice_split(dataset, dataset_name, "val",   repo_root=repo_root)
test  = slice_split(dataset, dataset_name, "test",  repo_root=repo_root)
print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

NameError: name 'slice_split' is not defined